# Lewis Signalling Games

## Language as Convention

Why does the word *dog* mean what it means? Not because of any intrinsic connection between sound and animal — it's a **convention**: a self-reinforcing pattern of behaviour that, once adopted by a community, becomes stable.

In 1969, philosopher David Lewis formalised this idea with a simple game that captures the minimal conditions for linguistic convention to emerge from scratch.

## The Game

```
  world  ──state──→  [ SENDER ]  ──message──→  [ RECEIVER ]  ──action──→ ?
                                                                   ↓
                              shared reward  ←───── (action == state?)
```

Each **round**:
1. The world draws a **state** $s \in \{0,\ldots,N{-}1\}$ uniformly at random
2. The **Sender** sees $s$ and emits a **message** $m \in \{0,\ldots,N{-}1\}$
3. The **Receiver** sees only $m$ and takes an **action** $a \in \{0,\ldots,N{-}1\}$
4. Both receive reward $r = +1$ if $a = s$, else $r = -1$

Neither agent can observe the other's internal state. A **signalling system** — a bijection between states and messages — is a Nash equilibrium that achieves $r = +1$ every round.

With $N = 4$ states there are $4! = 24$ signalling systems. Which one emerges is a matter of chance.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

rng = np.random.default_rng(42)

N_STATES   = 4   # number of world states  (= vocabulary size)
N_MESSAGES = 4
N_ROUNDS   = 4000  # total training rounds for the tabular game

## Tabular Roth–Erev Agents

### Learning rule

Each agent maintains a **weight matrix**. On every round the chosen (state, message) or (message, action) pair has its weight updated by the reward:

$$W[i, j] \leftarrow W[i, j] + r$$

Probabilities are computed with **softmax** over the rows:

$$P(j \mid i) = \frac{e^{W[i,j]}}{\sum_k e^{W[i,k]}}$$

Positive rewards increase the probability of the chosen action; negative rewards decrease it. This is the **Roth–Erev** rule, one of the simplest models of reinforcement learning.

All weights are initialised to a small constant $\varepsilon > 0$ to avoid division by zero at the start.

In [ ]:
class Sender:

    def __init__(self, n_states: int, n_messages: int, eps: float = 1e-6):
        self.n_messages = n_messages
        self.message_weights = np.full((n_states, n_messages), eps)
        self._last = (0, 0)  # (state, message) from last call

    def send_message(self, state: int) -> int:
        w = self.message_weights[state]
        w = w - w.max()  # subtract max for numerical stability before exp
        probs = np.exp(w) / np.exp(w).sum()
        message = rng.choice(self.n_messages, p=probs)
        self._last = (state, message)
        return message

    def learn_from_feedback(self, reward: int) -> None:
        self.message_weights[self._last] += reward


class Receiver:

    def __init__(self, n_messages: int, n_actions: int, eps: float = 1e-6):
        self.n_actions = n_actions
        self.action_weights = np.full((n_messages, n_actions), eps)
        self._last = (0, 0)  # (message, action) from last call

    def act(self, message: int) -> int:
        w = self.action_weights[message]
        w = w - w.max()  # subtract max for numerical stability before exp
        probs = np.exp(w) / np.exp(w).sum()
        action = rng.choice(self.n_actions, p=probs)
        self._last = (message, action)
        return action

    def learn_from_feedback(self, reward: int) -> None:
        self.action_weights[self._last] += reward


class World:

    def __init__(self, n_states: int):
        self.n_states = n_states
        self.state = 0

    def emit_state(self) -> int:
        self.state = rng.integers(self.n_states)
        return self.state

    def evaluate_action(self, action: int) -> int:
        return 1 if action == self.state else -1

### Task 1 — Training Loop

Complete the training loop below.

Each round:
1. The world emits a state *(given)*
2. The sender produces a message *(given)*
3. The receiver takes an action *(given)*
4. The world evaluates the action *(given)*
5. **Both agents learn from the reward** — call `learn_from_feedback` on each ← *your task*
6. **Track the reward** by appending it to `reward_history` ← *your task*

The loop also captures weight-matrix snapshots at fixed epochs for later visualisation (do not modify that part).

In [ ]:
sender   = Sender(N_STATES, N_MESSAGES)
receiver = Receiver(N_MESSAGES, N_STATES)
world    = World(N_STATES)

reward_history = []
snapshots = {}          # {epoch: {'sender': W, 'receiver': W}}
SNAP_AT = {0, N_ROUNDS // 4, N_ROUNDS // 2, N_ROUNDS - 1}

for epoch in range(N_ROUNDS):

    world_state = world.emit_state()
    message     = sender.send_message(world_state)
    action      = receiver.act(message)
    reward      = world.evaluate_action(action)

    # ── Task 1a: let both agents learn ────────────────────────────────────
    # YOUR CODE HERE

    # ── Task 1b: record the reward ────────────────────────────────────────
    # YOUR CODE HERE

    # Snapshot (do not modify)
    if epoch in SNAP_AT:
        snapshots[epoch] = {
            'sender':   sender.message_weights.copy(),
            'receiver': receiver.action_weights.copy(),
        }

print("Training complete.")
print(f"Final 200-round mean reward: {np.mean(reward_history[-200:]):.3f}  (perfect = 1.0)")

In [ ]:
# ── Learning curve ─────────────────────────────────────────────────────────
window = 100
smoothed = np.convolve(reward_history, np.ones(window) / window, mode='valid')

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(reward_history, alpha=0.15, color='steelblue', linewidth=0.5)
ax.plot(range(window - 1, N_ROUNDS), smoothed, color='steelblue', linewidth=2,
        label=f'{window}-round rolling mean')
ax.axhline(-1.0 + 2 / N_STATES, color='gray',  linestyle='--', linewidth=1,
           label=f'random baseline ({-1 + 2/N_STATES:.2f})')
ax.axhline(1.0,                  color='green', linestyle='--', linewidth=1,
           label='perfect signalling')
ax.set(xlabel='Round', ylabel='Reward', title='Part 1 — Tabular Roth–Erev')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Task 2 — Visualise the Signalling System

The weight matrices alone are hard to read; we need to convert them to probabilities first.

Complete `softmax_rows` so that it applies softmax **along each row** of the weight matrix,
returning a matrix of the same shape where each row sums to 1:

$$P[i, j] = \frac{e^{W[i,j]}}{\sum_k e^{W[i,k]}}$$

Then run the cell below to see snapshots of the emerging signalling system.

In [ ]:
def softmax_rows(W: np.ndarray) -> np.ndarray:
    """
    Apply row-wise softmax to weight matrix W.
    Returns an array of the same shape where each row is a probability distribution.
    """
    # ── Task 2: implement softmax over axis=1 ─────────────────────────────
    # YOUR CODE HERE
    raise NotImplementedError


# ── Quick check ───────────────────────────────────────────────────────────
test = np.array([[1.0, 2.0, 3.0]])
result = softmax_rows(test)
assert result.shape == test.shape, "Shape should be preserved"
assert abs(result.sum() - 1.0) < 1e-9, "Each row must sum to 1"
print("softmax_rows OK:", result.round(3))

In [ ]:
# ── Snapshot heatmaps ──────────────────────────────────────────────────────
epochs = sorted(snapshots.keys())
fig, axes = plt.subplots(2, len(epochs), figsize=(3.5 * len(epochs), 7))

for col, ep in enumerate(epochs):
    snap = snapshots[ep]

    sender_probs   = softmax_rows(snap['sender'])    # shape (N_STATES, N_MESSAGES)
    receiver_probs = softmax_rows(snap['receiver'])  # shape (N_MESSAGES, N_STATES)

    kw = dict(annot=True, fmt='.2f', cbar=False, vmin=0, vmax=1, square=True)

    sns.heatmap(sender_probs, ax=axes[0, col],
                xticklabels=[f'm{i}' for i in range(N_MESSAGES)],
                yticklabels=[f's{i}' for i in range(N_STATES)],
                cmap='Blues', **kw)
    axes[0, col].set(title=f'Sender — round {ep}',
                     xlabel='message', ylabel='state')

    sns.heatmap(receiver_probs, ax=axes[1, col],
                xticklabels=[f'a{i}' for i in range(N_STATES)],
                yticklabels=[f'm{i}' for i in range(N_MESSAGES)],
                cmap='Greens', **kw)
    axes[1, col].set(title=f'Receiver — round {ep}',
                     xlabel='action', ylabel='message')

plt.suptitle('Emerging signalling system (tabular)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Decode the emerged language ────────────────────────────────────────────
final_sender_probs   = softmax_rows(sender.message_weights)
final_receiver_probs = softmax_rows(receiver.action_weights)

state_to_msg = final_sender_probs.argmax(axis=1)
msg_to_action = final_receiver_probs.argmax(axis=1)

print("State  → message (sender argmax) :", state_to_msg)
print("Message → action  (receiver argmax):", msg_to_action)

composed = msg_to_action[state_to_msg]
is_valid = sorted(composed) == list(range(N_STATES))
print(f"\nState → action (composed): {composed}")
print(f"Valid signalling system: {is_valid}")

---
## Reflection Questions

1. **Arbitrariness of language.** Re-run the notebook with a different random seed (change `rng = np.random.default_rng(42)`). Do you get the same signalling system? What does this tell you about the relationship between form and meaning?

2. **Convergence speed.** How does the number of states $N$ affect the number of rounds needed to converge? Try `N_STATES = 6` and `N_STATES = 10`. Is the relationship linear or faster?

3. **Underspecified vocabulary.** Set `N_MESSAGES = 2` with `N_STATES = 4`. A perfect signalling system is now impossible — the sender cannot uniquely identify all states with only 2 signals. What is the theoretical maximum reward? What strategy do the agents adopt?

4. **Symmetry breaking.** At the start of training all weights are equal (initialised to $\varepsilon$). What breaks the symmetry and drives agents towards one particular signalling system rather than another?

5. **Negative reward.** The current reward scheme is $+1$ for success and $-1$ for failure. What would change if you used $0$ and $+1$ instead? Would the agents still learn? Would they learn faster or slower?